In [ ]:

import os
import csv
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Set style for publication-ready plots
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.dpi'] = 150


# ============================================================
# Get CSV filename based on algorithm
# ============================================================
def get_csv_filename(algorithm_name, seed):
    """
    Return the correct CSV filename for each algorithm.
    
    Args:
        algorithm_name: "TD3", "SAC", or "ACKTR"
        seed: seed value
    
    Returns:
        filename: name of the CSV file
    """
    if algorithm_name == 'TD3':
        return f"td3_training_seed_{seed}.csv"
    elif algorithm_name == 'SAC':
        return f"episode_metrics_SAC_seed_{seed}.csv"
    elif algorithm_name == 'ACKTR':
        return f"acktr_training_seed_{seed}.csv"
    else:
        return f"{algorithm_name.lower()}_training_seed_{seed}.csv"


# ============================================================
# Load results from all seeds for a given algorithm
# ============================================================
def load_seed_results(algorithm_name, base_log_dir, seeds):
    """
    Load episode metrics from all seeds for a given algorithm.
    
    Returns:
        all_rewards: list of arrays
        all_costs: list of arrays
        all_scrs: list of arrays
        all_cvis: list of arrays
    """
    all_rewards = []
    all_costs = []
    all_scrs = []
    all_cvis = []
    
    for seed in seeds:
        csv_filename = get_csv_filename(algorithm_name, seed)
        csv_path = os.path.join(base_log_dir, f"seed_{seed}", csv_filename)
        
        if not os.path.exists(csv_path):
            print(f"[WARNING] File not found: {csv_path}")
            continue
        
        rewards = []
        costs = []
        scrs = []
        cvis = []
        
        with open(csv_path, 'r') as f:
            reader = csv.DictReader(f)
            
            # Get column names
            fieldnames = reader.fieldnames
            print(f"[DEBUG] {algorithm_name} seed {seed} columns: {fieldnames}")
            
            for row in reader:
                try:
                    # Read reward
                    reward = None
                    if 'reward' in row:
                        reward = row['reward']
                    elif 'episode_reward' in row:
                        reward = row['episode_reward']
                    
                    if reward and reward.strip():
                        rewards.append(float(reward))
                    
                    # Read cost
                    cost = None
                    if 'total_cost' in row:
                        cost = row['total_cost']
                    elif 'cost' in row:
                        cost = row['cost']
                    
                    if cost and cost.strip():
                        costs.append(float(cost))
                    
                    # Read SCR (Self-Consumption Rate)
                    scr = None
                    if 'SCR' in row:
                        scr = row['SCR']
                    elif 'scr' in row:
                        scr = row['scr']
                    
                    if scr and scr.strip():
                        scrs.append(float(scr))
                    
                    # Read CVI (Comfort Violation Index)
                    cvi = None
                    if 'CVI' in row:
                        cvi = row['CVI']
                    elif 'cvi' in row:
                        cvi = row['cvi']
                    
                    if cvi and cvi.strip():
                        cvis.append(float(cvi))
                        
                except (ValueError, KeyError) as e:
                    print(f"[WARNING] Error parsing row: {e}")
                    continue
        
        if rewards:
            all_rewards.append(np.array(rewards))
            all_costs.append(np.array(costs) if costs else np.array(rewards) * 0)
            all_scrs.append(np.array(scrs) if scrs else np.array(rewards) * 0)
            all_cvis.append(np.array(cvis) if cvis else np.array(rewards) * 0)
            print(f"[LOADED] {algorithm_name} seed {seed}: {len(rewards)} episodes")
        else:
            print(f"[WARNING] {algorithm_name} seed {seed}: No valid data found")
    
    return all_rewards, all_costs, all_scrs, all_cvis


# ============================================================
# Compute mean and confidence intervals across seeds
# ============================================================
def compute_mean_and_ci(data_list, confidence=0.95):
    """
    Compute mean and confidence intervals across seeds.
    
    Returns:
        mean: mean across seeds at each episode
        lower: lower confidence bound
        upper: upper confidence bound
    """
    if not data_list:
        return None, None, None
    
    # Find minimum number of episodes across all seeds
    min_episodes = min([len(data) for data in data_list])
    
    # Truncate all arrays to same length
    truncated_data = [data[:min_episodes] for data in data_list]
    
    # Stack into 2D array [num_seeds x num_episodes]
    stacked = np.vstack(truncated_data)
    
    # Compute statistics
    mean = np.mean(stacked, axis=0)
    std = np.std(stacked, axis=0)
    
    # Compute confidence interval (using t-distribution)
    dof = len(data_list) - 1
    if dof > 0:
        t_value = stats.t.ppf((1 + confidence) / 2, dof)
    else:
        t_value = 1.96
    
    sem = std / np.sqrt(len(data_list))
    ci = t_value * sem
    
    return mean, mean - ci, mean + ci


# ============================================================
# Plot learning curves with confidence bands
# ============================================================
def plot_learning_curves(data_results, algorithm_names, ylabel, title, 
                          output_filename, ylim=None, smooth_window=10):
    """
    Plot learning curves for multiple algorithms with confidence bands.
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
    line_styles = ['-', '-', '-']
    
    has_data = False
    
    for i, (results, name) in enumerate(zip(data_results, algorithm_names)):
        mean, lower, upper = results
        
        if mean is None:
            print(f"[WARNING] No data for {name}")
            continue
        
        has_data = True
        
        # Apply moving average smoothing
        if smooth_window > 1 and len(mean) > smooth_window:
            mean_smooth = np.convolve(mean, np.ones(smooth_window)/smooth_window, mode='valid')
            lower_smooth = np.convolve(lower, np.ones(smooth_window)/smooth_window, mode='valid')
            upper_smooth = np.convolve(upper, np.ones(smooth_window)/smooth_window, mode='valid')
            episodes = np.arange(smooth_window, len(mean_smooth) + smooth_window)
        else:
            mean_smooth = mean
            lower_smooth = lower
            upper_smooth = upper
            episodes = np.arange(1, len(mean) + 1)
        
        # Plot mean line
        ax.plot(episodes, mean_smooth, color=colors[i], linestyle=line_styles[i], 
                linewidth=2, label=name)
        
        # Plot confidence band
        ax.fill_between(episodes, lower_smooth, upper_smooth, color=colors[i], alpha=0.2)
    
    if has_data:
        ax.set_xlabel('Training Episode', fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(title, fontsize=14)
        
        if ylim:
            ax.set_ylim(ylim)
        
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(output_filename, dpi=300, bbox_inches='tight')
        plt.savefig(output_filename.replace('.png', '.pdf'), bbox_inches='tight')
        print(f"[SAVED] {output_filename}")
        plt.show()
    else:
        print(f"[WARNING] No data available for {ylabel}")


# ============================================================
# Main: Generate all learning curves
# ============================================================
def main():
    # Define seeds (1 through 10)
    SEEDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    
    # Base directories
    base_dirs = {
        'TD3': './logs_TD3',
        'SAC': './logs_SAC',
        'ACKTR': './logs_ACKTR'
    }
    
    algorithms = ['TD3', 'SAC', 'ACKTR']
    
    # Load results for all algorithms
    all_rewards = {}
    all_costs = {}
    all_scrs = {}
    all_cvis = {}
    
    print("=" * 60)
    print("LOADING RESULTS FROM ALL SEEDS (1-10)")
    print("=" * 60)
    
    for algo in algorithms:
        print(f"\n[LOADING] {algo} results from: {base_dirs[algo]}")
        rewards, costs, scrs, cvis = load_seed_results(algo, base_dirs[algo], SEEDS)
        
        if rewards:
            all_rewards[algo] = rewards
            all_costs[algo] = costs
            all_scrs[algo] = scrs
            all_cvis[algo] = cvis
            print(f"[OK] Loaded {len(rewards)} seeds for {algo}")
        else:
            print(f"[WARNING] No data loaded for {algo}")
    
    # Create output directory
    output_dir = './figures'
    os.makedirs(output_dir, exist_ok=True)
    
    # Compute mean and CI for each algorithm for each metric
    print("\n" + "=" * 60)
    print("COMPUTING MEAN AND CONFIDENCE INTERVALS")
    print("=" * 60)
    
    reward_results = []
    cost_results = []
    scr_results = []
    cvi_results = []
    
    for algo in algorithms:
        # Reward
        if algo in all_rewards and all_rewards[algo]:
            mean_r, lower_r, upper_r = compute_mean_and_ci(all_rewards[algo])
            reward_results.append((mean_r, lower_r, upper_r))
            print(f"\n{algo} Reward - Final: {mean_r[-1]:.2f}" if mean_r is not None else f"\n{algo} Reward: No data")
        else:
            reward_results.append((None, None, None))
        
        # Cost
        if algo in all_costs and all_costs[algo]:
            mean_c, lower_c, upper_c = compute_mean_and_ci(all_costs[algo])
            cost_results.append((mean_c, lower_c, upper_c))
            print(f"{algo} Cost - Final: {mean_c[-1]:.2f}" if mean_c is not None else f"{algo} Cost: No data")
        else:
            cost_results.append((None, None, None))
        
        # SCR (Self-Consumption Rate)
        if algo in all_scrs and all_scrs[algo]:
            mean_s, lower_s, upper_s = compute_mean_and_ci(all_scrs[algo])
            scr_results.append((mean_s, lower_s, upper_s))
            print(f"{algo} SCR - Final: {mean_s[-1]:.3f}" if mean_s is not None else f"{algo} SCR: No data")
        else:
            scr_results.append((None, None, None))
        
        # CVI (Comfort Violation Index)
        if algo in all_cvis and all_cvis[algo]:
            mean_cvi, lower_cvi, upper_cvi = compute_mean_and_ci(all_cvis[algo])
            cvi_results.append((mean_cvi, lower_cvi, upper_cvi))
            print(f"{algo} CVI - Final: {mean_cvi[-1]:.2f}" if mean_cvi is not None else f"{algo} CVI: No data")
        else:
            cvi_results.append((None, None, None))
    
    # Generate all plots
    print("\n" + "=" * 60)
    print("GENERATING FIGURES")
    print("=" * 60)
    
    # Figure 1: Reward Curves
    plot_learning_curves(
        reward_results, algorithms,
        'Cumulative Reward per Episode',
        'Episode Reward Evolution (Mean ± 95% CI across 10 seeds)',
        os.path.join(output_dir, 'figure1_reward_curves.png'),
        smooth_window=10
    )
    
    # Figure 2: Cost Curves
    plot_learning_curves(
        cost_results, algorithms,
        'Total Operational Cost ($)',
        'Operational Cost Evolution (Mean ± 95% CI across 10 seeds)',
        os.path.join(output_dir, 'figure2_cost_curves.png'),
        smooth_window=10
    )
    
    # Figure 3: SCR Curves (Self-Consumption Rate)
    plot_learning_curves(
        scr_results, algorithms,
        'Self-Consumption Rate (SCR)',
        'SCR Evolution (Mean ± 95% CI across 10 seeds)',
        os.path.join(output_dir, 'figure3_scr_curves.png'),
        ylim=(0, 1),
        smooth_window=10
    )
    
    # Figure 4: CVI Curves (Comfort Violation Index)
    plot_learning_curves(
        cvi_results, algorithms,
        'Comfort Violation Index (CVI)',
        'CVI Evolution (Mean ± 95% CI across 10 seeds)',
        os.path.join(output_dir, 'figure4_cvi_curves.png'),
        smooth_window=10
    )
    
    print("\n[SUCCESS] All learning curve plots generated successfully!")
    print(f"[INFO] Figures saved to: {output_dir}")
    print("\n" + "=" * 60)
    print("FIGURE SUMMARY FOR MANUSCRIPT")
    print("=" * 60)
    print("Figure 1: Episode Reward Evolution (all three algorithms together)")
    print("Figure 2: Operational Cost Evolution (all three algorithms together)")
    print("Figure 3: Self-Consumption Rate (SCR) Evolution (all three algorithms together)")
    print("Figure 4: Comfort Violation Index (CVI) Evolution (all three algorithms together)")
    print("\nAll figures show:")
    print("  - Solid lines: mean across 10 seeds")
    print("  - Shaded regions: 95% confidence interval")
    print("  - X-axis: Training episode (0-500)")


if __name__ == "__main__":
    main()